# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [1]:
%idle_timeout 2880
%glue_version 5.0
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.8 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 5.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 2880
Session ID: 72412a79-2e9b-4a0c-811b-892432df7562
Applying the following default arguments:
--glue_kernel_version 1.0.8
--enable-glue-datacatalog true
Waiting for session 72412a79-2e9b-4a0c-811b-892432df7562 to get into ready status...
Session 72412a79-2e9b-4a0c-811b-892432df7562 ha

#### Example: Create a DynamicFrame from a table in the AWS Glue Data Catalog and display its schema


In [2]:
sellers_dyf = glueContext.create_dynamic_frame.from_catalog(database='brazilian_ecommerce_raw', table_name='sellers')

sellers_dyf.printSchema()

root
|-- seller_id: string
|-- seller_zip_code_prefix: long
|-- seller_city: string
|-- seller_state: string


#### Example: Convert the DynamicFrame to a Spark DataFrame and display a sample of the data


In [3]:
sellers_df = sellers_dyf.toDF()
sellers_df_sample = sellers_df.limit(1000)

/usr/lib/spark/python/lib/pyspark.zip/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.


In [5]:
# Step 1: Drop rows with null value in seller_id column
cleaned_df = sellers_df_sample.dropna(subset=["seller_id"])

# Step 2: Standardize city names and state codes
from pyspark.sql.functions import lower, upper, trim, col

cleaned_df = cleaned_df.withColumn("seller_city", lower(trim(cleaned_df["seller_city"])))
cleaned_df = cleaned_df.withColumn("seller_state", upper(trim(cleaned_df["seller_state"])))

# Step 3: Drop rows with duplicate seller_id columns
cleaned_df = cleaned_df.dropDuplicates(["seller_id"])

# Step 4: Validate state codes
brazil_codes = ["AC","AL","AP","AM","BA","CE","DF","ES","GO","MA","MT","MS","MG",
               "PA","PB","PR","PE","PI","RJ","RN","RS","RO","RR","SC","SP","SE","TO"]

cleaned_df = cleaned_df.filter(col("seller_state").isin(brazil_codes))

In [6]:
# Convert the pyspark Dataframe back into a DynamicFrame
from awsglue.dynamicframe import DynamicFrame

cleaned_dyf = DynamicFrame.fromDF(cleaned_df, glueContext, "cleaned_dyf")

In [7]:
# Store the cleaned dataset as Parquet in S3
s3output = glueContext.getSink(
  path="s3://bucket181rt2/clean/sellers",
  connection_type="s3",
  updateBehavior="UPDATE_IN_DATABASE",
  partitionKeys=[],
  compression="snappy",
  enableUpdateCatalog=True,
  transformation_ctx="s3output",
)
s3output.setCatalogInfo(
  catalogDatabase="brazilian_ecommerce_clean", catalogTableName="sellers"
)
s3output.setFormat("glueparquet")
s3output.writeFrame(cleaned_dyf)